In [7]:
from pathlib import Path
import pandas as pd

INPUT_ROOT = Path("../PARP")
OUTPUT_CSV = Path("../clean_data/parp_raw_combined.csv")
SHEET_NAME = "OAPARP61-B"

GROUP_HEADERS = {
    "WELL TEST DATE",
    "WELL STATUS",
    "WELL TEST DATA",
    "WELL TEST",
}
SUBHEADER_TOKENS = {
    "MTH",
    "DAY",
    "YR",
    "HRS",
    "CPR",
    "TPR",
    "SEP",
    "BOPD",
    "BWPD",
    "GOR",
}


def normalize_col_name(value):
    text = "" if pd.isna(value) else str(value)
    text = " ".join(text.replace("\n", " ").split())
    return text.strip()


def build_headers(row):
    headers = []
    for idx, value in enumerate(row):
        name = "" if pd.isna(value) else str(value).strip()
        if not name or name.lower() == "nan":
            name = f"Column{idx + 1}"
        headers.append(name)
    return headers


def make_unique_columns(columns):
    counts = {}
    unique_cols = []
    for col in columns:
        base = str(col)
        if base in counts:
            counts[base] += 1
            unique_cols.append(f"{base}.{counts[base]}")
        else:
            counts[base] = 0
            unique_cols.append(base)
    return unique_cols


def is_subheader_row(row):
    values = {
        normalize_col_name(v).upper()
        for v in row.tolist()
        if pd.notna(v) and normalize_col_name(v)
    }
    return bool(values & SUBHEADER_TOKENS)


def merge_headers(primary, secondary):
    merged = list(primary)
    for idx, sub in enumerate(secondary):
        sub_clean = normalize_col_name(sub)
        if not sub_clean or sub_clean.lower().startswith("column"):
            continue
        prim_clean = normalize_col_name(primary[idx]).upper()
        if not prim_clean or prim_clean.startswith("COLUMN") or prim_clean in GROUP_HEADERS:
            merged[idx] = sub_clean
    return merged


def locate_header_index(df):
    for idx, row in df.iterrows():
        values = {
            normalize_col_name(v).upper()
            for v in row.tolist()
            if pd.notna(v) and normalize_col_name(v)
        }
        has_field = "FIELD" in values or "FIELD NAME" in values
        has_resvr = "RESVR" in values or "RESVR CODE" in values
        has_date = (
            "TEST YEAR" in values
            or "YR" in values
            or "TEST MONTH" in values
            or "WELL TEST DATE" in values
            or "WELL TEST DAY" in values
        )
        if has_field and has_resvr and has_date:
            return idx
    return None


def normalize_year_value(value):
    if pd.isna(value):
        return pd.NA
    text = str(value).strip()
    if not text:
        return pd.NA
    try:
        number = int(float(text))
    except ValueError:
        return value
    if number > 60:
        return 1900 + number
    if number < 27:
        return 2000 + number
    return number


def load_sheet(path):
    engine = "xlrd" if path.suffix.lower() == ".xls" else "openpyxl"
    with pd.ExcelFile(path, engine=engine) as xl:
        if SHEET_NAME not in xl.sheet_names:
            return pd.DataFrame(), None
        df = pd.read_excel(xl, sheet_name=SHEET_NAME, header=None, dtype=str)

    header_idx = locate_header_index(df)
    if header_idx is None:
        return pd.DataFrame(), SHEET_NAME

    df = df.loc[header_idx:].copy()
    header_row = df.iloc[0]
    subheader_row = df.iloc[1] if len(df) > 1 else None

    headers = build_headers(header_row)
    if subheader_row is not None and is_subheader_row(subheader_row):
        subheaders = build_headers(subheader_row)
        headers = merge_headers(headers, subheaders)
        data = df.iloc[2:].copy()
    else:
        data = df.iloc[1:].copy()

    headers = [normalize_col_name(h) for h in headers]
    data.columns = make_unique_columns(headers)
    data = data.dropna(how="all")

    for col in ["YR", "YEAR"]:
        if col in data.columns:
            data[col] = data[col].apply(normalize_year_value)

    if "WELL" in data.columns:
        well_text = data["WELL"].astype(str).str.strip()
        data = data[data["WELL"].notna() & (well_text != "") & (well_text.str.lower() != "nan")]

    data.insert(0, "sheet", SHEET_NAME)
    data.insert(0, "source", path.name)

    return data, SHEET_NAME


files = sorted([*INPUT_ROOT.rglob("*.xls"), *INPUT_ROOT.rglob("*.xlsx")])
print("files", len(files))

all_data = []
for path in files:
    df, sheet = load_sheet(path)
    if df.empty:
        print(f"No data found in {path.name} ({sheet})")
        continue
    all_data.append(df)

combined = pd.concat(all_data, ignore_index=True) if all_data else pd.DataFrame()
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
combined.to_csv(OUTPUT_CSV, index=False)

print(f"Wrote {len(combined)} rows to {OUTPUT_CSV}")
combined.head()

files 107
No data found in 12 DEC 2001 PARP REPORTS.xls (OAPARP61-B)
Wrote 61918 rows to ..\clean_data\parp_raw_combined.csv


,source,sheet,FIELD,RESVR,RESERVOIR,WELL,MTH,DAY,YR,HRs,...,PERC,GTY &,64S,****************W E L L T E S T D A T A ****************,TPR,SEP,BOPD,BWPD,GOR,Column19
0,02 FEB 1996 PARP REPORTS.xls,OAPARP61-B,BAHI,BARG,BAHI ARGUB SAND STONE,A-064-32,1,12,1996,95.1,...,0,44.3,0,NaN,66,63,880,0,33,...
1,02 FEB 1996 PARP REPORTS.xls,OAPARP61-B,BAHI,BAUK,BAHI UPER KALASH,KK-001-32,2,15,1996,96.4,...,23.5,44.9,0,NaN,64,58,724,222,383,...
2,02 FEB 1996 PARP REPORTS.xls,OAPARP61-B,BAHI,BPL7,PALEOCENE,A-006-32,2,2,1996,96.1,...,61.3,44.9,0,NaN,65,58,260,412,165,...
3,02 FEB 1996 PARP REPORTS.xls,OAPARP61-B,BAHI,BPL7,PALEOCENE,A-008-32,2,7,1996,96.1,...,70.4,44.4,0,NaN,78,62,1016,2412,165,...
4,02 FEB 1996 PARP REPORTS.xls,OAPARP61-B,BAHI,BPL7,PALEOCENE,A-009-32,2,6,1996,96.1,...,72.4,44.9,0,NaN,65,58,172,452,165,...


In [8]:
cols_to_drop = ["...", "****************W E L L T E S T D A T A ****************", "Column19","sheet"]
existing = [c for c in cols_to_drop if c in combined.columns]
combined = combined.drop(columns=existing, errors="ignore")

OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
combined.to_csv(OUTPUT_CSV, index=False)

print("Dropped columns:", existing)
print(f"Wrote {len(combined)} rows to {OUTPUT_CSV}")
combined.head()

Dropped columns: ['****************W E L L T E S T D A T A ****************', 'Column19', 'sheet']
Wrote 61918 rows to ..\clean_data\parp_raw_combined.csv


,source,FIELD,RESVR,RESERVOIR,WELL,MTH,DAY,YR,HRs,WELL.1,PERC,GTY &,64S,TPR,SEP,BOPD,BWPD,GOR
0,02 FEB 1996 PARP REPORTS.xls,BAHI,BARG,BAHI ARGUB SAND STONE,A-064-32,1,12,1996,95.1,PSP,0,44.3,0,66,63,880,0,33
1,02 FEB 1996 PARP REPORTS.xls,BAHI,BAUK,BAHI UPER KALASH,KK-001-32,2,15,1996,96.4,PTN,23.5,44.9,0,64,58,724,222,383
2,02 FEB 1996 PARP REPORTS.xls,BAHI,BPL7,PALEOCENE,A-006-32,2,2,1996,96.1,PSP,61.3,44.9,0,65,58,260,412,165
3,02 FEB 1996 PARP REPORTS.xls,BAHI,BPL7,PALEOCENE,A-008-32,2,7,1996,96.1,PSP,70.4,44.4,0,78,62,1016,2412,165
4,02 FEB 1996 PARP REPORTS.xls,BAHI,BPL7,PALEOCENE,A-009-32,2,6,1996,96.1,PSP,72.4,44.9,0,65,58,172,452,165


In [9]:
date_parts = ["MTH", "DAY", "YR"]
missing = [c for c in date_parts if c not in combined.columns]

if missing:
    print("Missing date columns:", missing)
else:
    date_text = (
        combined["DAY"].fillna("").astype(str).str.strip()
        + "/"
        + combined["MTH"].fillna("").astype(str).str.strip()
        + "/"
        + combined["YR"].fillna("").astype(str).str.strip()
    )
    combined["TEST DATE"] = pd.to_datetime(date_text, errors="coerce", dayfirst=True).dt.date

OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
combined.to_csv(OUTPUT_CSV, index=False)

print("TEST DATE created")
combined[["MTH", "DAY", "YR", "TEST DATE"]].head()

C:\Users\OPS010\AppData\Local\Temp\ipykernel_14388\1493281762.py:12: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  + combined["YR"].fillna("").astype(str).str.strip()


TEST DATE created


,MTH,DAY,YR,TEST DATE
0,1,12,1996,1996-01-12
1,2,15,1996,1996-02-15
2,2,2,1996,1996-02-02
3,2,7,1996,1996-02-07
4,2,6,1996,1996-02-06


In [10]:
rename_map = {
    "WELL.1": "WELL TYPE",
    "GTY &": "GRAVITY",
    "BOPD": "DAILY OIL PRODUCTION",
    "BWPD": "WATER PRODUCTION",
    "PERC": "BSW",
}

existing = {k: v for k, v in rename_map.items() if k in combined.columns}
combined = combined.rename(columns=existing)

OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
combined.to_csv(OUTPUT_CSV, index=False)

print("Renamed:", existing)
combined.head()

Renamed: {'WELL.1': 'WELL TYPE', 'GTY &': 'GRAVITY', 'BOPD': 'DAILY OIL PRODUCTION', 'BWPD': 'WATER PRODUCTION', 'PERC': 'BSW'}


,source,FIELD,RESVR,RESERVOIR,WELL,MTH,DAY,YR,HRs,WELL TYPE,BSW,GRAVITY,64S,TPR,SEP,DAILY OIL PRODUCTION,WATER PRODUCTION,GOR,TEST DATE
0,02 FEB 1996 PARP REPORTS.xls,BAHI,BARG,BAHI ARGUB SAND STONE,A-064-32,1,12,1996,95.1,PSP,0,44.3,0,66,63,880,0,33,1996-01-12
1,02 FEB 1996 PARP REPORTS.xls,BAHI,BAUK,BAHI UPER KALASH,KK-001-32,2,15,1996,96.4,PTN,23.5,44.9,0,64,58,724,222,383,1996-02-15
2,02 FEB 1996 PARP REPORTS.xls,BAHI,BPL7,PALEOCENE,A-006-32,2,2,1996,96.1,PSP,61.3,44.9,0,65,58,260,412,165,1996-02-02
3,02 FEB 1996 PARP REPORTS.xls,BAHI,BPL7,PALEOCENE,A-008-32,2,7,1996,96.1,PSP,70.4,44.4,0,78,62,1016,2412,165,1996-02-07
4,02 FEB 1996 PARP REPORTS.xls,BAHI,BPL7,PALEOCENE,A-009-32,2,6,1996,96.1,PSP,72.4,44.9,0,65,58,172,452,165,1996-02-06


In [11]:
drop_cols = ["MTH", "DAY", "YR"]
combined = combined.drop(columns=[c for c in drop_cols if c in combined.columns], errors="ignore")

OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
combined.to_csv(OUTPUT_CSV, index=False)

print("Dropped:", drop_cols)
print(f"Wrote {len(combined)} rows to {OUTPUT_CSV}")
combined.head()

Dropped: ['MTH', 'DAY', 'YR']
Wrote 61918 rows to ..\clean_data\parp_raw_combined.csv


,source,FIELD,RESVR,RESERVOIR,WELL,HRs,WELL TYPE,BSW,GRAVITY,64S,TPR,SEP,DAILY OIL PRODUCTION,WATER PRODUCTION,GOR,TEST DATE
0,02 FEB 1996 PARP REPORTS.xls,BAHI,BARG,BAHI ARGUB SAND STONE,A-064-32,95.1,PSP,0,44.3,0,66,63,880,0,33,1996-01-12
1,02 FEB 1996 PARP REPORTS.xls,BAHI,BAUK,BAHI UPER KALASH,KK-001-32,96.4,PTN,23.5,44.9,0,64,58,724,222,383,1996-02-15
2,02 FEB 1996 PARP REPORTS.xls,BAHI,BPL7,PALEOCENE,A-006-32,96.1,PSP,61.3,44.9,0,65,58,260,412,165,1996-02-02
3,02 FEB 1996 PARP REPORTS.xls,BAHI,BPL7,PALEOCENE,A-008-32,96.1,PSP,70.4,44.4,0,78,62,1016,2412,165,1996-02-07
4,02 FEB 1996 PARP REPORTS.xls,BAHI,BPL7,PALEOCENE,A-009-32,96.1,PSP,72.4,44.9,0,65,58,172,452,165,1996-02-06


In [12]:
for col in combined.select_dtypes(include="object").columns:
    combined[col] = combined[col].astype(str).str.strip()

OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
combined.to_csv(OUTPUT_CSV, index=False)

print("Trimmed whitespace in text columns")
combined.head()

Trimmed whitespace in text columns


,source,FIELD,RESVR,RESERVOIR,WELL,HRs,WELL TYPE,BSW,GRAVITY,64S,TPR,SEP,DAILY OIL PRODUCTION,WATER PRODUCTION,GOR,TEST DATE
0,02 FEB 1996 PARP REPORTS.xls,BAHI,BARG,BAHI ARGUB SAND STONE,A-064-32,95.1,PSP,0,44.3,0,66,63,880,0,33,1996-01-12
1,02 FEB 1996 PARP REPORTS.xls,BAHI,BAUK,BAHI UPER KALASH,KK-001-32,96.4,PTN,23.5,44.9,0,64,58,724,222,383,1996-02-15
2,02 FEB 1996 PARP REPORTS.xls,BAHI,BPL7,PALEOCENE,A-006-32,96.1,PSP,61.3,44.9,0,65,58,260,412,165,1996-02-02
3,02 FEB 1996 PARP REPORTS.xls,BAHI,BPL7,PALEOCENE,A-008-32,96.1,PSP,70.4,44.4,0,78,62,1016,2412,165,1996-02-07
4,02 FEB 1996 PARP REPORTS.xls,BAHI,BPL7,PALEOCENE,A-009-32,96.1,PSP,72.4,44.9,0,65,58,172,452,165,1996-02-06
